# 调试单个来源

客户的周报格式变了、或者想试一个新规则时，用这个本子只跑一家，看清楚每一步。
**这里调用的函数和正式运行（`python -m psi_tool.run`）是同一套**，试好之后把改动写回 `config/sources.yaml` 即可，不要在本子里另写一套算法。

从上往下依次运行（Shift+Enter）。

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_rows", 200, "display.max_columns", 30, "display.width", 200)
from psi_tool.sources import load_config, load_source, candidates, read_sheet, find_header
from psi_tool.mapping import Mapper, latest_ttl
from psi_tool.run import build, summarize, compare
from psi_tool.paths import CONFIG, MAPPING, PSI_FILE, TTL_DIR

CONFIG = load_config("config/sources.yaml")
print("可选来源：", list(CONFIG))

## 1. 选来源和周

In [ ]:
SOURCE = "AUCHAN"        # 改成要调试的来源名（见上面的列表）
WEEK = 44
INBOX = "W44测试/原始文件"    # 历史数据在 数据/历史周报
cfg = dict(CONFIG[SOURCE])
cfg

## 2. 能找到哪些候选文件

没找到：检查 `file` 正则和 `week_in_path`。找到多个：检查是不是有你处理过的副本。

In [ ]:
for p in candidates(INBOX, cfg, WEEK):
    print(p)

## 3. 看原始表（没有任何处理）

用来确认表头在第几行、列名有没有变。

In [ ]:
path = candidates(INBOX, cfg, WEEK)[0]
raw = read_sheet(path, cfg.get("sheet", 0))
print("表头在第", find_header(raw, [h.replace("{week}", str(WEEK)) for h in cfg["header"]]), "行（从 0 数）")
raw.head(15)

## 4. 按配置读出来的明细（已筛选品类）

可以直接改上面的 `cfg`（例如 `cfg["include"] = ...`）再重跑这一格做试验。

In [ ]:
res = load_source(INBOX, SOURCE, cfg, WEEK)
print("文件：", res.files); print("时间段：", res.period); print("提示：", res.warnings)
print("文件里全部行合计 SO / 库存：", res.raw_total_so, res.raw_total_stock)
print("筛选后合计 SO / 库存：", res.rows["so"].sum(), res.rows["stock"].sum())
res.rows

## 5. 映射结果

`status`：psi = 匹配到 PSI 行；others = 有代码但 PSI 没这一行；unmatched = 查不到代码；excluded = 映射表标记排除。

In [ ]:
results, mapper = build(WEEK, INBOX, str(PSI_FILE), str(CONFIG), str(MAPPING), str(TTL_DIR))
sheet = res.psi_sheet
rows = results[sheet]["rows"]
rows = rows[rows["source"] == SOURCE]
display(rows.groupby("status")[["so", "stock"]].sum())
rows[rows["status"] != "psi"]

## 6. 和 PSI 里已填的这一周比对

In [ ]:
table = summarize(results[sheet])
checked, diffs = compare(results[sheet], table, WEEK)
print(f"{sheet}: 比对 {checked} 格，不一致 {len(diffs)}")
pd.DataFrame(diffs)